# Case study: missing prices, missing ratings and extreme prices of Berlin Airbnb listings

*Session 4, block 2 · Author: course team · Licence: CC-BY-4.0*

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared with
`uv run python case-study/prepare_airbnb.py`. The data were collected from public listing pages; host names and
texts have been removed. Report results in aggregate and do not try to identify hosts.

**Questions of the practice block**

1. A third of the listings have no price. Is a missing price related to the number of reviews, or to something
   else? What does that say about the missing-data mechanism (MCAR, MAR, MNAR)?
2. Why is the review score missing for one listing in five, and why should it not be imputed?
3. How well do simple, KNN and iterative imputation recover the **number of bedrooms** when we hide known values
   on purpose?
4. Which prices and minimum stays are outliers by the IQR rule, the z-score and the MAD rule, and which of the
   flagged values are errors?

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run: uv run python case-study/prepare_airbnb.py"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

listings = pd.read_parquet(DATA / "listings.parquet")
print(f"{len(listings):,} listings, snapshot {listings['last_scraped'].min():%Y-%m-%d}")
listings.isna().mean().sort_values(ascending=False).head(12).round(3)

## 1. Is a missing price related to the number of reviews?

A **missingness indicator** is 1 where the value is missing. Comparing it across other variables shows whether
the listings with and without a price differ. If the share of missing prices varies with an observed variable,
**MCAR** (missing completely at random) is implausible.

In [ ]:
listings["price_missing"] = listings["price"].isna()
recent = pd.cut(listings["number_of_reviews_ltm"], [-1, 0, 2, 10, 10_000],
                labels=["0", "1-2", "3-10", ">10"])
listings.groupby(recent, observed=True)["price_missing"].agg(share_missing="mean", n="size").round(3)

Listings without a review in the last twelve months lack a price ten times as often as busy listings. Before
concluding that reviews "cause" missing prices, look for a third variable. Inside Airbnb reads the price from the
listing page for a bookable date; a listing that has **no free night** in the coming year shows no price.

In [ ]:
blocked = listings["availability_365"].eq(0)
print(f"no free night in the next 365 days: {blocked.mean():.1%} of listings")
print(listings.groupby(blocked)["price_missing"].mean().round(3))
active = listings[~blocked]
print("among listings with at least one free night:")
print(active.groupby(active["number_of_reviews_ltm"].eq(0))["price_missing"].agg(["mean", "size"]).round(3))

**A test of independence.** The chi-square test asks whether the missing share is the same in all review bands
(null hypothesis: independence). With 12,776 listings, the p-value will be tiny; look at the size of the
differences as well (applied recap from the statistics module).

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(recent, listings["price_missing"])
chi2, p_value, dof, _ = chi2_contingency(table)
print(table)
print(f"chi2 = {chi2:.0f}, dof = {dof}, p = {p_value:.1e}")

**Can the observed columns predict missingness?** If a classifier predicts the indicator better than chance
(area under the ROC curve, AUC, above 0.5), the data are not MCAR, and the columns it uses belong in any
imputation model. (Logistic regression and the AUC are taught in Sessions 6 and 8; here we only read the
number.)

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = pd.DataFrame({
    "blocked": blocked.astype(int),
    "log_reviews_ltm": np.log1p(listings["number_of_reviews_ltm"]),
    "log_min_nights": np.log(listings["minimum_nights"].fillna(listings["minimum_nights"].median())),
    "accommodates": listings["accommodates"],
    "room_type": listings["room_type"],
})
for cols in (["log_reviews_ltm"], ["blocked"], list(features.columns)):
    numeric = [c for c in cols if c != "room_type"]
    categorical = [c for c in cols if c == "room_type"]
    model = make_pipeline(
        make_column_transformer((OneHotEncoder(handle_unknown="ignore"), categorical), (StandardScaler(), numeric)),
        LogisticRegression(max_iter=1000))
    auc = cross_val_score(model, features[cols], listings["price_missing"], cv=5, scoring="roc_auc").mean()
    print(f"AUC {auc:.2f}  using {cols}")

**Interpretation.** A missing price is clearly not MCAR. Most of it is **structural**: a listing with no
bookable night has no current price at all (99.9 % missing), and such listings also collect few reviews. That is
why the number of reviews alone predicts missingness well (AUC about 0.79) but availability predicts it better
(about 0.90). Among listings that *can* be booked, the share without a price is still 20 % when there were no
recent reviews against 4 % otherwise: dormant listings often show no price. Whether those dormant listings would
be priced differently (MNAR) cannot be decided from these data.

**Consequence for analysis.** Do not impute the price of blocked listings: there is no price to recover. Analyse
prices for the listings that can be booked, and say so ("prices of the 8,441 listings with a current price").

**Exercise.** Compare the median `accommodates`, the share of entire homes and the district mix of listings with
and without a price. Would the median price of all listings be higher or lower if the missing prices were known?
Argue, do not impute.

## 2. Review scores: missing by design

A listing that has never been reviewed cannot have a rating. This is **structural missingness**: the value does
not exist, so imputing a rating would invent information. The fact "no reviews yet" is itself a useful
variable.

In [ ]:
no_reviews = listings["number_of_reviews"].eq(0)
print(pd.crosstab(no_reviews.rename("no reviews"), listings["review_scores_rating"].isna().rename("rating missing")))
print(listings.groupby(no_reviews)[["availability_365", "minimum_nights", "accommodates"]].median())

Every listing without a rating has no reviews, and every listing with reviews has a rating. Listings without
reviews have more free nights and longer minimum stays: many are new or medium-term offers. Keep the rating
missing and add the indicator `no_reviews`.

## 3. Imputation compared on hidden values: bedrooms

`bedrooms` is missing for 27 % of listings: for 70 % of private rooms (the field is often left empty when one room
is rented) and for 9 % of entire homes. To compare imputers fairly we need the truth, so we take the entire homes
**with** a known number of bedrooms, hide 30 % of them, impute, and compare with the hidden values. Predictors:
the number of guests (`accommodates`) and the distance to the centre (Alexanderplatz).

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

def km_to_centre(lat, lon, lat0=52.5219, lon0=13.4132):
    """Distance in km to Alexanderplatz (haversine formula)."""
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = np.sin((lat0 - lat) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon0 - lon) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

homes = listings[listings["room_type"].eq("Entire home/apt") & listings["bedrooms"].notna()]
X = pd.DataFrame({"accommodates": homes["accommodates"], "bedrooms": homes["bedrooms"],
                  "km_to_centre": km_to_centre(homes["latitude"], homes["longitude"])})
print(len(X), "entire homes with known bedrooms")
print(X.corr().round(2))

rng = np.random.default_rng(0)
hide = rng.random(len(X)) < 0.3
truth = X.loc[hide, "bedrooms"].to_numpy()
X_missing = X.copy()
X_missing.loc[hide, "bedrooms"] = np.nan

In [ ]:
def evaluate(filled):
    filled = np.asarray(filled)
    return {"RMSE": np.sqrt(np.mean((filled - truth) ** 2)), "exact after rounding": np.mean(np.round(filled) == truth)}

results = {}
for name, imputer in [("mean", SimpleImputer(strategy="mean")),
                      ("median", SimpleImputer(strategy="median")),
                      ("iterative", IterativeImputer(random_state=0))]:
    results[name] = evaluate(imputer.fit_transform(X_missing)[hide, 1])

scaler = StandardScaler().fit(X_missing)                      # KNN needs comparable scales
knn = KNNImputer(n_neighbors=10).fit_transform(scaler.transform(X_missing))
results["KNN (k = 10)"] = evaluate(scaler.inverse_transform(knn)[hide, 1])

by_guests = X_missing.groupby("accommodates")["bedrooms"].transform("median")   # a simple domain rule
results["median for the same number of guests"] = evaluate(by_guests[hide])
pd.DataFrame(results).T.round(3)

**Interpretation.** The mean and the median ignore the size of the flat; the median (1 bedroom) is exactly right
for most flats but badly wrong for large ones. Methods that use the number of guests roughly halve the error,
because guests and bedrooms are strongly correlated (r about 0.73). The distance to the centre adds almost
nothing. A one-line domain rule (median bedrooms for the same number of guests) is about as good as the
iterative imputer and easier to explain.

**Exercise.** Repeat the comparison with `beds` added as a predictor (it is itself missing for half of the
listings). Does the iterative imputer gain? Why should an imputer be fitted on training data only?

## 4. Univariate outliers: price and minimum stay

Three rules: the IQR rule (Tukey's fences), the z-score (|z| > 3) and the robust z-score with the median absolute
deviation (|z\*| > 3.5).

In [ ]:
def flag_rules(x: pd.Series) -> pd.DataFrame:
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    z = (x - x.mean()) / x.std()
    mad = (x - x.median()).abs().median()
    robust_z = 0.6745 * (x - x.median()) / mad
    return pd.DataFrame({"IQR": (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr),
                         "z": z.abs() > 3, "MAD": robust_z.abs() > 3.5})

price = listings["price"].dropna()
print(price.describe().round(1))
print("share flagged, raw scale:", flag_rules(price).mean().round(3).to_dict())
print("share flagged, log scale:", flag_rules(np.log10(price)).mean().round(3).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
ax.hist(np.log10(price), bins=80, color="#a9c7ee")
ax.set_xticks([0, 1, 2, 3, 4], ["1", "10", "100", "1,000", "10,000"])
ax.set(xlabel="EUR per night (log scale)", ylabel="listings", title="Two peaks: what is the low one?")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

The log scale shows a **second peak** around €20 per night. Look at who is there before flagging anything.

In [ ]:
priced = listings[listings["price"].notna()].copy()
priced["stay"] = pd.cut(priced["minimum_nights"], [0, 6, 27, 91, 10_000],
                        labels=["1-6 nights", "7-27", "28-91", "92+"])
print(priced.groupby(["stay", "room_type"], observed=True)["price"].agg(["size", "median"]).round(1))
print("listings under 10 EUR with a minimum stay of 28+ nights:",
      f"{(priced.loc[priced['price'] < 10, 'minimum_nights'] >= 28).mean():.0%}")

**Interpretation.** Almost all very low prices belong to listings with a minimum stay of at least 28 nights
(most often exactly 92). An entire flat for about €23 a night is far below any holiday price in Berlin and well
below a typical furnished monthly rent. The price field of these medium-term listings is therefore **not
comparable** with the price of short stays; we cannot tell from the data how it was computed. This is not an
outlier to delete one by one; it is a different population. Session 5 and Session 6 analyse prices of
short-stay listings (minimum stay below 28 nights) and say so.

At the top end, the IQR rule flags 4 % of prices on the raw scale, all of them high: a consequence of the skewed
distribution. A handful of short-stay listings ask several thousand euros a night.

In [ ]:
top = listings.nlargest(6, "price")[["price", "property_type", "room_type", "accommodates", "minimum_nights",
                                        "availability_365", "number_of_reviews"]]
top

Prices of €8,000 and €10,025 a night for a house for five and a loft for seven guests are implausible for
their size. They may be typing errors or deliberately high prices that keep a listing from being booked; the data
cannot tell which. Other high prices are genuine (look at `property_type`: houseboats for 12 to 16
guests). Flag them, keep them out of
averages, and use robust methods (Session 6).

**Minimum and maximum stay.**

In [ ]:
nights = listings["minimum_nights"].dropna()
print(nights.quantile([0.5, 0.75, 0.9, 0.99]).to_dict(), "max", nights.max())
print("share with exactly 92 nights:", round((nights == 92).mean(), 3))
mad = (nights - nights.median()).abs().median()
print("median", nights.median(), "MAD", mad, "-> robust z flags",
      round((0.6745 * (nights - nights.median()).abs() / mad > 3.5).mean(), 3))
print(listings["maximum_nights"].value_counts().head(4).to_dict())
print("maximum_nights = 2,147,483,647:", (listings["maximum_nights"] == 2**31 - 1).sum())

**Interpretation.** The minimum stay has two clusters (1–3 nights and 92 nights), so every one-centre rule
misleads: the MAD rule flags the whole medium-term cluster (37 % of listings). A domain rule is clearer:
"minimum stay of 28 nights or more = medium-term listing". One listing asks for at least 1,125 nights (three
years), the platform maximum; keep it but flag it. In `maximum_nights`, the value 2,147,483,647 = 2³¹ − 1 is the
largest 32-bit integer: a software default, not a stay. Set it to missing.

**Exercise.** Write a log entry (rule, rows, action, reason) for each finding in this notebook. Workbook 15
turns them into the cleaning pipeline for the listings.